# Antimicrobial Peptide Classification against *Klebsiella pneumoniae*
## ESM-2 embeddings + ProtParam + modlAMP descriptors  ->  Machine Learning  ->  TreeSHAP

**Dataset** `natural-modified-combined.csv` - natural + modified/synthetic peptides, ~600 rows, binary `Active` / `Inactive` MIC labels.

| Stage | Content |
|---|---|
| 0 | Environment (Colab) |
| 1 | **Which ML family?** Options, trade-offs and a recommendation for n ~ 600 |
| 2 | Data loading + cleaning (mojibake units, non-canonical residues, conflicting duplicate rows) |
| 3 | Leakage control: exact dedup + CD-HIT style 90% identity clustering |
| 4 | Feature extraction 2.3.1 - ESM-2 (HuggingFace) pooled embeddings, cached |
| 5 | Feature extraction 2.3.2a - ProtParam physico-chemical descriptors |
| 6 | Feature extraction 2.3.2b - modlAMP descriptors (net charge, hydrophobic moment, amphipathicity) |
| 7 | Feature integration 2.3.3 - three blocks: Descriptors / ESM-2 / Combined |
| 8 | Model zoo 2.4.1 - 13 classifiers as leakage-free Pipelines |
| 9 | Train/test ladder 100/0 -> 10/90 (proposal point 1) with repeated seeds |
| 10 | Metrics: Accuracy, **MCC**, **AUROC**, AUPRC, F1, Sens, Spec, BACC |
| 11 | Hyper-parameter optimization (randomized search + 5-fold CV) for the top models |
| 12 | Final model, 5-fold CV, bootstrap CIs, headline table |
| 13 | SHAP 2.4.2 - TreeSHAP global + per-residue attribution mapped onto sequence positions |
| 14 | Artefacts and a `predict_peptides()` inference helper |
| 15 | (Optional) LoRA fine-tuning of ESM-2 with a classification head |

> **Runtime tip** Colab > Runtime > Change runtime type > **T4 GPU**. Steps 2, 3, 5, 6 are CPU only; step 4 is much faster on GPU.
>
> **Headline caveat** this dataset contains the *same peptide* listed twice and a few sequences labelled both Active and Inactive. Cell 3 fixes that; without it every metric below is inflated by several MCC points.


In [ ]:
# @title 0. Install packages
import sys, subprocess, importlib


def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)


_pip('pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn', 'biopython', 'joblib')
_pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
_pip('transformers', 'accelerate', 'sentencepiece')
_pip('modlamp')
_pip('xgboost', 'lightgbm', 'catboost')
_pip('shap', 'optuna')

for m in ('numpy', 'pandas', 'sklearn', 'Bio', 'torch', 'transformers', 'shap', 'xgboost', 'lightgbm'):
    try:
        mod = importlib.import_module(m)
        print(f'{m:12s} OK   {getattr(mod, "__version__", "?")}')
    except Exception as e:
        print(f'{m:12s} FAIL {e}')

try:
    import modlamp
    from modlamp.descriptors import GlobalDescriptor, PeptideDescriptor
    print('modlamp      OK  ', modlamp.__version__)
except Exception as e:
    print('modlamp      FAIL (independent fallback descriptors will be used):', e)

import torch
print('CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('-> enable GPU: Colab > Runtime > Change runtime type > GPU')


# 1. Which machine-learning approach? (n ~ 600, d = 1280)

With ~600 samples and 1280-dimensional ESM-2 embeddings, the **bias-variance trade-off is the whole game**: anything with high capacity memorises the training set and collapses MCC on held-out data. The families below are ordered from *safest* to *most powerful* for this regime.

## 1.1 Options

| # | Family | Concrete models | Why it fits n ~ 600 | Expected MCC / AUROC | Risk |
|---|---|---|---|---|---|
| A | **Regularised linear** | Logistic Regression (L2 / ElasticNet), Linear SVM | n << d is exactly the regime L2-regularised linear models were designed for; lowest variance; calibrated probabilities so AUROC is trustworthy | 0.45-0.60 / 0.80-0.88 | under-fits long-range position effects |
| B | **Kernel SVM** | SVC (RBF) on scaled features | usually the strongest *classical single model* in small-n AMP/QSAR benchmarks; tolerates p >> n | 0.55-0.70 / 0.85-0.92 | needs `C` / `gamma` tuning, slow, probabilities need Platt scaling |
| C | **Tree ensembles** (the proposal) | **Random Forest**, ExtraTrees, GradientBoosting, HistGradientBoosting, XGBoost, LightGBM, CatBoost | non-linear, no scaling needed, native TreeSHAP, robust against wide embedding vectors | 0.55-0.70 / 0.85-0.92 | at d = 1280 trees mostly see noise, so the ESM-2 block **must** be compressed (SVD/PLS) first |
| D | **Discriminant / naive Bayes** | LDA with shrinkage, GaussianNB, ComplementNB, PLS-DA | almost no overfitting, seconds to fit, a defensible baseline row | 0.35-0.55 / 0.75-0.87 | weak non-linear capacity |
| E | **Instance based** | k-NN (k = 5-15) on reduced embeddings | interpretable, zero training cost, answers "does this resemble a known active peptide?" | 0.30-0.50 / 0.65-0.80 | collapses on the 10/90 split |
| F | **Neural** | MLP, 1-2 hidden layers, heavy weight decay + early stopping | small enough for 600 rows if regularised; mirrors the head used when fine-tuning a PLM | 0.50-0.65 / 0.83-0.90 | unstable on very small training fractions |
| G | **Ensembles (expected best)** | soft Voting / Stacking of SVC-RBF + RF/ExtraTrees + XGB/LGBM + LogReg | averaging decorrelated errors usually gives the best *and most stable* MCC and AUROC; a stacking meta-learner on out-of-fold probabilities is the natural top model at n ~ 600 | **0.60-0.75 / 0.88-0.94** | several times the compute; must be trained on OOF predictions |
| H | **PLM fine-tuning** | ESM-2 + LoRA (or last-k-layer) fine-tune + linear head | highest ceiling: the encoder adapts to AMP grammar (aromatic spacing, cationic periodicity, beta-sheet propensity) | 0.65-0.80 / 0.90-0.96 | needs GPU, 15-40 min, high seed variance - only defensible with repeated nested CV |

## 1.2 Recommendation for this dataset

1. **Core results table = A, C, G** (LogReg, SVC-RBF, RF, ExtraTrees, XGBoost/LightGBM, soft-voting ensemble). Keep D as a sanity baseline, E as a 90/10 vs 10/90 sensitivity check, F optional, H only as a clearly labelled extension.
2. **Compress the ESM-2 block inside the CV loop** (`TruncatedSVD`/`SafeSVD`, 64-256 components) for tree models; give the full 1280-d vector to SVC/LogReg after `StandardScaler`. Never fit the SVD on the whole dataset - that is leakage.
3. **Report the ablation** Descriptors vs ESM-2 vs Combined. Descriptors alone (~90 features, d << n) are often within noise of the PLM block; if so, the cheap interpretable model is the more defensible one.
4. **Optimise MCC, not accuracy.** On a ~50/50 dataset accuracy saturates near 0.85 and hides failure. Use MCC for model selection and report AUROC + AUPRC + MCC + balanced accuracy + Sens/Spec.
5. **De-duplicate and cluster-group the splits.** The file holds exact duplicate sequences with *conflicting* labels and near-identical clusters of up to 4 members. The notebook evaluates random **and** cluster-grouped splits so the published numbers survive review.
6. **Never tune on the test set.** All hyper-parameter search happens by 5-fold CV *inside the training portion*; the outer split is scored once.


In [ ]:
# @title 2. Configuration
import os, json, math, random, warnings, time, hashlib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = '/content/amp_klebsiella'
os.makedirs(WORKDIR, exist_ok=True)

# ---------------- data ----------------
CSV_CANDIDATES = [
    '/content/drive/MyDrive/Klebsiella pneumoniae/natural-modified-combined.csv',
    '/content/natural-modified-combined.csv',
    '/content/drive/MyDrive/natural-modified-combined.csv',
]

# ---------------- leakage control ----------------
DEDUP_EXACT = True             # collapse identical sequences
DROP_LABEL_CONFLICTS = True    # drop sequences recorded both Active and Inactive
CLUSTER_IDENTITY = 0.90        # CD-HIT style identity for near-duplicate groups
BUILD_CLUSTERS = True
GROUP_AWARE_SPLIT = True       # a cluster never straddles the train/test border

# ---------------- ESM-2 (section 2.3.1) ----------------
ESM_MODEL = 'facebook/esm2_t33_650M_UR50D'   # t12_35M fast | t30_150M middle | t33_650M best
# ESM_MODEL = 'facebook/esm2_t30_150M_UR50D'
# ESM_MODEL = 'facebook/esm2_t12_35M_UR50D'
ESM_BATCH = 16
ESM_POOL = 'mean'              # 'mean' | 'cls' | 'mean+cls'
ESM_MAX_LEN = 256

# ---------------- feature integration (section 2.3.3) ----------------
USE_DESCRIPTORS = True
USE_ESM = True
SVD_COMPONENTS = 128           # compression of the ESM-2 block for tree models
SHOW_PCA = True

# ---------------- evaluation ----------------
SPLIT_PAIRS = [(1.00, 0.00), (0.90, 0.10), (0.80, 0.20), (0.70, 0.30), (0.60, 0.40),
               (0.50, 0.50), (0.40, 0.60), (0.30, 0.70), (0.20, 0.80), (0.10, 0.90)]
# NOTE: the 8th ratio in the proposal was written 30/40; it is read here as 30 train / 70 test.
SEEDS = [42, 7, 123, 2024, 99]         # full evaluation
SCREEN_SEEDS = [42, 7, 123]            # cheaper model screening
CV_FOLDS = 5
CV_REPEATS = 2                         # only used by the 100/0 (no test set) case
TUNING_N_ITER = 40                     # randomized search iterations per model
PRIMARY_METRIC = 'mcc'                 # what the hyper-parameter search maximises
N_BOOTSTRAP = 2000
CV_REPORT_BLOCKS = ['Descriptors', 'ESM2', 'Combined']

print('config ok | workdir:', WORKDIR)
print('n splits:', len(SPLIT_PAIRS), '| seeds:', SEEDS)


# 3. Load the dataset

Put the CSV in Google Drive (for example `MyDrive/Klebsiella pneumoniae/`) and run the mount cell, or use the upload fallback. The file holds two stacked blocks - natural peptides with CD-HIT cluster columns, then DBAASP synthetic/modified peptides - separated by one empty row.


In [ ]:
# @title 3.1 Mount Google Drive (or upload the CSV)
from google.colab import drive, files
import os

drive.mount('/content/drive')


def resolve_csv(candidates):
    for p in candidates:
        if os.path.exists(p):
            print('found:', p)
            return p
    print('none of the configured paths exist -> falling back to the file uploader')
    up = files.upload()
    for name in up:
        p = os.path.join('/content', name)
        print('uploaded:', p)
        return p
    raise FileNotFoundError('CSV not found and nothing was uploaded')


CSV_PATH = resolve_csv(CSV_CANDIDATES)


In [ ]:
# @title 3.2 Raw load and inspection
raw = pd.read_csv(CSV_PATH, encoding='latin-1', encoding_errors='replace')
raw.columns = [str(c).strip() for c in raw.columns]

print('shape:', raw.shape)
print('columns:', list(raw.columns))
print()
print(raw.head(3).to_string())
print()
for c in ['Activity', 'Activity Measure', 'Medium', 'synthesis type (for modified)']:
    if c in raw.columns:
        print('--- value counts:', c)
        print(raw[c].value_counts(dropna=False).to_string())
        print()
if 'sequence' in raw.columns:
    print('unique raw sequences:', raw['sequence'].nunique())
    print('length range:', raw['sequence'].dropna().astype(str).str.len().min(),
          '-', raw['sequence'].dropna().astype(str).str.len().max())


In [ ]:
# @title 3.3 Clean labels, sequences, non-canonical residues, provenance
STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')
AA_ALIAS = {'U': 'C', 'O': 'K', 'B': 'X', 'Z': 'X', 'J': 'X', 'X': 'X'}


def clean_seq(s):
    # uppercase, drop whitespace and gap characters, map non-canonical residues
    if not isinstance(s, str):
        return '', ''
    raw_s = s.strip().upper().replace(' ', '')
    mapped, noncanon = [], []
    for ch in raw_s:
        if ch in '-*.':
            continue
        if ch in STANDARD_AA:
            mapped.append(ch)
        else:
            noncanon.append(ch)
            mapped.append(AA_ALIAS.get(ch, 'X'))
    return ''.join(mapped), ''.join(sorted(set(noncanon)))


def clean_label(v):
    if not isinstance(v, str):
        return np.nan
    v = v.strip().lower()
    if v.startswith('activ'):
        return 1
    if v.startswith('inactiv') or v.startswith('not activ'):
        return 0
    return np.nan


df = raw.copy()
seq_clean = df['sequence'].map(clean_seq)
df['sequence'] = [s for s, _ in seq_clean]
df['noncanonical_residues'] = [n for _, n in seq_clean]
df['n_noncanonical'] = df['noncanonical_residues'].map(len)
df['label'] = df['Activity'].map(clean_label)
df['length'] = df['sequence'].str.len()

df['pep_id'] = df['Peptide ID'].astype(str).str.strip()
df['source_block'] = np.where(df['pep_id'].str.match(r'^DBAASP', case=False), 'modified_DBAASP', 'natural')
df['strain'] = df['Target Species'].astype(str)
df['synthesis_type'] = df['synthesis type (for modified)'].fillna('natural').astype(str).str.strip().str.lower()
df['medium'] = df['Medium'].astype(str).str.strip()
df['unit'] = df['Unit'].astype(str).str.strip()

n0 = len(df)
df = df.dropna(subset=['sequence', 'label'])
df = df[df['sequence'].str.len() >= 5]
df['label'] = df['label'].astype(int)
df = df.reset_index(drop=True)

print(f'rows: {n0} -> {len(df)}  (dropped {n0 - len(df)} unusable)')
print('label counts:')
print(df['label'].value_counts().to_string())
print('class balance: %.1f%% active' % (100 * df['label'].mean()))
print('source block:')
print(df['source_block'].value_counts().to_string())
print('synthesis type:')
print(df['synthesis_type'].value_counts().to_string())
print('medium:')
print(df['medium'].value_counts().to_string())
print('length: min %d  median %d  max %d' % (df['length'].min(), df['length'].median(), df['length'].max()))
print('sequences with non-canonical residues:', int((df['n_noncanonical'] > 0).sum()))
if (df['n_noncanonical'] > 0).any():
    print(df.loc[df['n_noncanonical'] > 0, 'noncanonical_residues'].value_counts().to_string())
print()
print('label x source block:')
print(pd.crosstab(df['source_block'], df['label']).to_string())


# 4. Leakage control: duplicate sequences and label conflicts

The natural block and the DBAASP block contain overlapping records, and a few sequences carry *both* labels (e.g. the same peptide is `Inactive` in one record and `Active` in another, and two records for peptide 4294 disagree). With a random split such a pair is trivially memorised, which inflates every metric.

* **4.1** collapse exact duplicates (majority label), drop conflicting sequences, and keep a `n_records` count.
* **4.2** build CD-HIT style groups at 90% identity with a 3-mer prefilter plus Biopython local alignment, so `GROUP_AWARE_SPLIT` can keep a group on one side of the split.
* **4.3** drop clusters that still contain both labels.


In [ ]:
# @title 4.1 Exact duplicate consolidation
before = len(df)

if DEDUP_EXACT:
    agg = df.groupby('sequence')['label'].agg(['nunique', 'mean', 'size'])
    conflict_seqs = set(agg.index[agg['nunique'] > 1])
    print(f'{int((agg["size"] > 1).sum())} sequences occur more than once; '
          f'{len(conflict_seqs)} of them carry conflicting labels')
    if conflict_seqs:
        show = df[df['sequence'].isin(conflict_seqs)][['pep_id', 'source_block', 'sequence', 'label']]
        print(show.sort_values('sequence').to_string(index=False))
    if DROP_LABEL_CONFLICTS and conflict_seqs:
        df = df[~df['sequence'].isin(conflict_seqs)].copy().reset_index(drop=True)
        print('dropped', len(conflict_seqs), 'conflicting sequences')
        agg = df.groupby('sequence')['label'].agg(['nunique', 'mean', 'size'])
    keep = agg['mean'].round().astype(int).to_frame('label')
    keep['n_records'] = agg['size']
    meta = df.groupby('sequence').agg(
        pep_id=('pep_id', 'first'),
        source_block=('source_block', 'first'),
        strain=('strain', 'first'),
        synthesis_type=('synthesis_type', 'first'),
        medium=('medium', 'first'),
        unit=('unit', 'first'),
        length=('length', 'first'),
        noncanonical_residues=('noncanonical_residues', 'first'),
        n_noncanonical=('n_noncanonical', 'first'),
    )
    df = meta.join(keep).reset_index()

df['label'] = df['label'].astype(int)
df = df.reset_index(drop=True)

print(f'rows: {before} -> {len(df)} | unique sequences: {df["sequence"].nunique()}')
print('active / inactive:', int(df['label'].sum()), '/', int((1 - df['label']).sum()))
df.head()


In [ ]:
# @title 4.2 CD-HIT style clustering (k-mer prefilter + Biopython alignment)
from Bio import Align
import time

K = 3


def kmer_vec(seq, k=K):
    if len(seq) >= k:
        return set(seq[i:i + k] for i in range(len(seq) - k + 1))
    return {seq}


def identity_aligned(a_seq, b_seq, aligner, min_len):
    # local alignment identity = identical aligned positions / shortest sequence
    # NOTE: Aln.aligned yields (start, end) coordinate pairs per block, not strings,
    # so the aligned substrings are sliced out of the originals.
    aln = aligner.align(a_seq, b_seq)[0]
    blocks_a, blocks_b = aln.aligned
    matched = 0
    for (a0, a1), (b0, b1) in zip(blocks_a, blocks_b):
        matched += sum(1 for x, y in zip(a_seq[a0:a1], b_seq[b0:b1]) if x == y)
    return matched / max(1, min_len)


def build_clusters(seqs, identity=0.90):
    t0 = time.time()
    n = len(seqs)
    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.open_gap_score = -11
    aligner.extend_gap_score = -1
    aligner.substitution_matrix = Align.substitution_matrices.load('BLOSUM62')

    kmers = [kmer_vec(s) for s in seqs]
    order = np.argsort([-len(s) for s in seqs])      # longest first -> cluster representatives
    clusters = np.full(n, -1, dtype=int)
    rep_idx, rep_km, cid = [], [], 0
    n_align = 0
    prefilter = max(0.5, identity - 0.35)            # cheap gate before the real alignment
    for i in order:
        si, li = seqs[i], len(seqs[i])
        ki = kmers[i]
        placed = False
        for k, r in enumerate(rep_idx):
            lr = len(seqs[r])
            if abs(li - lr) / max(li, lr) > (1 - identity):
                continue                              # length filter
            if len(ki & rep_km[k]) / max(1, min(len(ki), len(rep_km[k]))) < prefilter:
                continue                              # cheap k-mer prefilter
            n_align += 1
            if identity_aligned(si, seqs[r], aligner, min(li, lr)) >= identity:
                clusters[i] = clusters[r]
                placed = True
                break
        if not placed:
            clusters[i] = cid
            rep_idx.append(i)
            rep_km.append(ki)
            cid += 1
    print('clustered %d sequences at identity >= %.2f: %d clusters (%d redundant), '
          '%d real alignments, %.1fs' % (n, identity, cid, n - cid, n_align, time.time() - t0))
    return clusters


if BUILD_CLUSTERS:
    df['cluster'] = build_clusters(df['sequence'].tolist(), CLUSTER_IDENTITY)
else:
    df['cluster'] = np.arange(len(df))

print('cluster size distribution (size : n_clusters)')
print(df['cluster'].value_counts().value_counts().sort_index().to_string())


In [ ]:
# @title 4.3 Drop clusters with mixed labels, then persist
bad = set(df.groupby('cluster')['label'].nunique().pipe(lambda s: s[s > 1]).index)
if bad:
    print('dropping', len(bad), 'clusters that still contain both labels')
    print(df[df['cluster'].isin(bad)][['pep_id', 'cluster', 'sequence', 'label']]
          .sort_values('cluster').to_string(index=False))
    df = df[~df['cluster'].isin(bad)].copy().reset_index(drop=True)

    print('FINAL modelling set: n = %d | active = %d | inactive = %d | clusters = %d | largest cluster = %d'
          % (len(df), int(df['label'].sum()), int((1 - df['label']).sum()),
             df['cluster'].nunique(), int(df['cluster'].value_counts().max())))
    print('balance: %.1f%% active' % (100 * df['label'].mean()))
    print('length: min %d  median %d  max %d' % (df['length'].min(), df['length'].median(), df['length'].max()))

seqs = df['sequence'].tolist()
df.to_csv(os.path.join(WORKDIR, '01_dataset_clean.csv'), index=False)


# 5. Feature extraction 2.3.1 - ESM-2 protein language model embeddings (HuggingFace)

Every peptide is tokenised with the ESM-2 tokenizer (residue order preserved), passed through the pretrained transformer, and reduced to a **single 1280-d vector** by masked mean pooling over residue tokens (`ESM_POOL='mean'`; `'cls'` and `'mean+cls'` are available for comparison). The `<cls>`/`<eos>` tokens and padding are excluded from the pooling mask. Results are cached to `.npz`, so re-running the notebook after changing the split logic costs nothing.

Checkpoint choice: `esm2_t12_35M_UR50D` (fast, 480-d), `esm2_t30_150M_UR50D` (640-d), `esm2_t33_650M_UR50D` (1280-d, the proposal's d = 1280). Longer peptides (ribosomal fragments) are truncated at `ESM_MAX_LEN`.


In [ ]:
# @title 5.1 ESM-2 embedding extraction (cached)
import torch
from transformers import AutoTokenizer, AutoModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


@torch.no_grad()
def esm2_embed(sequences, model_name=ESM_MODEL, pool=ESM_POOL, batch_size=ESM_BATCH,
               max_len=ESM_MAX_LEN, device=DEVICE, verbose=True):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).eval().to(device)
    if device == 'cuda':
        model.half()

    order = np.argsort([-len(s) for s in sequences])        # length-sorted batching
    vecs = [None] * len(sequences)
    t0 = time.time()
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        enc = tok([sequences[i] for i in idx], return_tensors='pt',
                  padding=True, truncation=True, max_length=max_len)
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc).last_hidden_state                 # (B, L, D)
        mask = enc['attention_mask'].clone()
        mask[(mask == tok.cls_token_id) | (mask == tok.eos_token_id)] = 0
        if pool == 'cls':
            v = out[:, 0, :].float().cpu().numpy()
        else:
            m = mask.unsqueeze(-1).float()
            mean = (out.float() * m).sum(1) / m.sum(1).clamp(min=1)
            v = mean.cpu().numpy()
            if pool == 'mean+cls':
                v = np.concatenate([v, out[:, 0, :].float().cpu().numpy()], axis=1)
        for j, i in enumerate(idx):
            vecs[i] = v[j]
        if verbose and (start // batch_size) % 10 == 0:
            print('  %d/%d  (%.0fs)' % (min(start + batch_size, len(order)), len(order), time.time() - t0),
                  flush=True)
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()
    X = np.vstack(vecs).astype(np.float32)
    print('ESM-2 %s -> %s in %.0fs' % (model_name, X.shape, time.time() - t0))
    return X


cache_key = hashlib.md5(('|'.join(seqs) + ESM_MODEL + ESM_POOL + str(ESM_MAX_LEN)).encode()).hexdigest()[:10]
cache = os.path.join(WORKDIR, 'esm2_%s.npz' % cache_key)

if os.path.exists(cache) and USE_ESM:
    X_esm = np.load(cache)['X']
    print('loaded cache', cache, X_esm.shape)
else:
    X_esm = esm2_embed(seqs)
    np.savez_compressed(cache, X=X_esm, sequences=np.array(seqs), model=ESM_MODEL)

ESM_DIM = X_esm.shape[1]
print('ESM matrix:', X_esm.shape, '| nan:', int(np.isnan(X_esm).sum()), '| inf:', int(np.isinf(X_esm).sum()))


# 6. Feature extraction 2.3.2a - ProtParam physico-chemical properties

Contains the four properties named in the proposal - **instability index** (values > 40 flag peptides that are likely unstable in vitro/in vivo and therefore expensive synthesis mistakes), **aliphatic index** (thermostability proxy), **theoretical pI** (cross-checked against the modlAMP charge in the next cell) and **GRAVY** (hydrophilic/hydrophobic balance; ultra-hydrophobic sequences that precipitate during assay prep are flagged) - plus molecular weight, net charge, aromaticity, extinction coefficients, secondary-structure fractions, per-residue composition and a pH-dependent charge profile.


In [ ]:
# @title 6.1 ProtParam descriptors
from Bio.SeqUtils.ProtParam import ProteinAnalysis

PH_CURVE = [5.0, 5.5, 6.0, 6.5, 7.0, 7.4, 7.8, 8.0, 8.5, 9.0]


def aliphatic_index(seq):
    f = ProteinAnalysis(seq).count_amino_acids()
    return 100.0 * (f['A'] + 2.9 * f['V'] + 3.9 * (f['I'] + f['L'])) / max(1, len(seq))


def protparam_features(seq):
    a = ProteinAnalysis(seq)
    c = a.count_amino_acids()
    L = max(1, len(seq))
    red, ox = a.molar_extinction_coefficient()
    helix, turn, sheet = a.secondary_structure_fraction()
    f = {}
    f['length'] = len(seq)
    f['molecular_weight'] = a.molecular_weight()
    f['aromaticity'] = a.aromaticity()
    f['instability_index'] = a.instability_index()
    f['theoretical_pI'] = a.isoelectric_point()
    f['gravy'] = a.gravy()
    f['aliphatic_index'] = aliphatic_index(seq)
    f['charge_pH7.4'] = a.charge_at_pH(7.4)
    f['frac_KR'] = (c['K'] + c['R']) / L
    f['frac_KRKHKRH'] = (c['K'] + c['R'] + c['H']) / L
    f['frac_pos_10'] = (c['K'] + c['R'] + 0.1 * c['H']) / L
    f['frac_DE'] = (c['D'] + c['E']) / L
    f['frac_WYF'] = (c['W'] + c['Y'] + c['F']) / L
    f['frac_G'] = c['G'] / L
    f['frac_C'] = c['C'] / L
    f['frac_P'] = c['P'] / L
    f['n_C'] = c['C']
    f['n_basic'] = c['K'] + c['R'] + c['H']
    f['n_acidic'] = c['D'] + c['E']
    f['n_polar'] = c['S'] + c['T'] + c['N'] + c['Q'] + c['C']
    f['n_hydrophobic'] = c['A'] + c['V'] + c['L'] + c['I'] + c['M'] + c['F'] + c['W'] + c['Y'] + c['P']
    f['frac_hydrophobic'] = f['n_hydrophobic'] / L
    f['hydrophobic_ratio'] = f['n_hydrophobic'] / max(1, f['n_polar'])
    f['n_strong_hydrophobic'] = c['L'] + c['I'] + c['V'] + c['F'] + c['W'] + c['M']
    f['n_C_term5'] = int(seq[-5:].count('C'))
    f['n_N_term5'] = int(seq[:5].count('C'))
    f['molar_ext_coeff_reduced'] = red
    f['molar_ext_coeff_oxidised'] = ox
    f['frac_helix'] = helix
    f['frac_turn'] = turn
    f['frac_sheet'] = sheet
    for aa in 'ACDEFGHIKLMNPQRSTVWY':
        f['AA_%s' % aa] = c[aa]
    for ph in PH_CURVE:
        f['charge_pH%.1f' % ph] = a.charge_at_pH(ph)
    f['flag_unstable_II_gt40'] = int(f['instability_index'] > 40)
    f['flag_ultra_hydrophobic'] = int(f['gravy'] > 1.0)
    return f


X_desc = pd.DataFrame([protparam_features(s) for s in seqs])
PROTPARAM_COLS = list(X_desc.columns)

key_cols = ['length', 'molecular_weight', 'instability_index', 'aliphatic_index',
            'theoretical_pI', 'gravy', 'charge_pH7.4', 'frac_KR', 'frac_DE', 'frac_WYF']
print('ProtParam matrix:', X_desc.shape)
print(X_desc[key_cols].describe().T.round(3).to_string())
print()
print('flagged unstable (instability index > 40):', int(X_desc['flag_unstable_II_gt40'].sum()),
      '| flagged ultra-hydrophobic (GRAVY > 1):', int(X_desc['flag_ultra_hydrophobic'].sum()))


# 7. Feature extraction 2.3.2b - modlAMP AMP-specific descriptors

* **Net charge** - ability to interact with the negatively charged bacterial membrane.
* **Hydrophobic moment** - amphipathicity, the driver of membrane disruption.
* **Amphipathicity** - balance of hydrophobic and hydrophilic regions.

The cell calls `modlamp.descriptors` (Eisenberg consensus scale) **and** an independent Kugelberg / Kyte-Doolittle helical-wheel implementation, so the table is produced even if the modlamp wheel fails on short peptides. Where both charges exist they are cross-checked, which is the consistency check described in the proposal (ProtParam pI vs modlamp charge).


In [ ]:
# @title 7.1 modlAMP descriptors (+ independent fallback implementations)
import tempfile

KD = dict(zip('AVILPFLMYWSTGC', [1.8, 4.2, 4.5, 3.8, -1.6, 2.8, -0.9, 1.9, -0.4, -0.8, -0.7, -0.1, -1.3, -0.5, 1.9, -0.8]))
EI_raw = dict(zip('AVLIPFMDWKSTYRNQCEG',
                  [0.62, 0.46, 0.30, 0.37, 0.26, 0.35, 0.21, 0.10, 0.61, 0.42, 0.29,
                   0.27, 0.28, 0.14, 0.61, 0.25, 0.15, 0.10, 0.28, 0.29]))
_mu, _sd = float(np.mean(list(EI_raw.values()))), float(np.std(list(EI_raw.values())))
EI = {k: (v - _mu) / _sd for k, v in EI_raw.items()}
CHARGE_PH = {**{k: 0.0 for k in 'ACDEFGHILNPQSTVWY'}, 'K': 1.0, 'R': 1.0, 'H': 0.1, 'D': -1.0, 'E': -1.0}
TANFORD = {'L': 4.92, 'I': 4.92, 'V': 3.67, 'F': 2.98, 'M': 2.35, 'W': 2.33, 'A': 1.81,
           'C': 0.24, 'G': 0.74, 'T': 0.78, 'S': 0.64, 'Y': 0.76, 'P': 0.64, 'H': 0.0, 'E': 0.0,
           'D': 0.0, 'N': 0.0, 'Q': 0.0, 'K': 0.0, 'R': 0.0}


def helicoidal_moment(seq, scale, window=11):
    # hydrophobic moment on a helical wheel (3.6 residues / turn), max over all windows
    v = np.array([scale.get(a, 0.0) for a in seq], dtype=float)
    n = len(v)
    if n == 0:
        return 0.0
    w = min(window, n)
    ang = 2 * np.pi * np.arange(w) / w
    best = 0.0
    for s in range(n - w + 1):
        seg = v[s:s + w]
        mu = np.hypot((seg * np.cos(ang)).sum(), (seg * np.sin(ang)).sum()) / w
        best = max(best, mu)
    return float(best)


def modlamp_features(seq):
    n = max(1, len(seq))
    v = np.array([KD.get(a, 0.0) for a in seq], dtype=float)
    f = {}
    f['ml_net_charge_pH7'] = float(sum(CHARGE_PH.get(a, 0.0) for a in seq))
    f['ml_charge_density'] = f['ml_net_charge_pH7'] / n
    f['ml_hydrophobic_moment'] = helicoidal_moment(seq, KD, 11)
    f['ml_hydrophobic_moment_eis'] = helicoidal_moment(seq, EI, 11)
    f['ml_mean_hydropathy'] = float(v.mean())
    f['ml_max_hydropathy_window'] = float(v.max())
    f['ml_min_hydropathy_window'] = float(v.min())
    f['ml_hydropathy_range'] = float(v.max() - v.min())
    f['ml_hydrophobicity'] = float(v.mean())
    f['ml_amphipathicity'] = f['ml_hydrophobic_moment'] / (abs(v.mean()) + 0.1)
    f['ml_amphipathicity_norm'] = f['ml_hydrophobic_moment'] / (0.1 + float(np.abs(v).mean()))
    f['ml_boman_index'] = float(np.mean([1.0 if a in 'AILVFWMY' else 0.0 for a in seq]))
    f['ml_tanford_G'] = float(sum(TANFORD.get(a, 0.0) for a in seq))
    f['ml_autocorr_lag3_kd'] = float(np.corrcoef(v[:-3], v[3:])[0, 1]) if len(v) > 4 else 0.0
    f['ml_autocorr_lag4_kd'] = float(np.corrcoef(v[:-4], v[4:])[0, 1]) if len(v) > 5 else 0.0
    win = 7
    f['ml_frac_cationic_window'] = float(max(
        [sum(1.0 for a in seq[i:i + win] if a in 'KR') / max(1, len(seq[i:i + win]))
         for i in range(max(1, len(seq) - win + 1))] or [0.0]))
    return f


def modlamp_native(seqs):
    # native modlamp descriptors, one fresh descriptor object per quantity.
    # NOTE: GlobalDescriptor/PeptideDescriptor reset their internal matrix whenever
    # a method is called with append=True on a fresh object, so a single object
    # cannot accumulate columns in modlamp 4.3 -> one object per descriptor.
    try:
        from modlamp.descriptors import GlobalDescriptor, PeptideDescriptor
        tmp = tempfile.NamedTemporaryFile('w', suffix='.fasta', delete=False)
        for i, s in enumerate(seqs):
            tmp.write('>%d\n%s\n' % (i, s))
        tmp.close()
        n = len(seqs)
        blocks = {}

        def grab(obj, attr):
            arr = np.asarray(getattr(obj, attr), dtype=float)
            if arr.size % n != 0:
                raise ValueError('unexpected descriptor size %s' % (arr.shape,))
            return arr.reshape(n, -1)[:, 0]

        for meth in ('length', 'boman_index', 'aromaticity', 'aliphatic_index',
                     'instability_index', 'hydrophobic_ratio', 'charge_density',
                     'isoelectric_point'):
            g = GlobalDescriptor(tmp.name)
            getattr(g, meth)()
            blocks['mlAMP_' + meth] = grab(g, 'descriptor')
        g = GlobalDescriptor(tmp.name)
        g.calculate_charge(ph=7.0)
        blocks['mlAMP_net_charge_pH7'] = grab(g, 'descriptor')
        g = GlobalDescriptor(tmp.name)
        g.calculate_MW()
        blocks['mlAMP_MW'] = grab(g, 'descriptor')
        p = PeptideDescriptor(tmp.name, 'eisenberg')
        p.calculate_global()
        blocks['mlAMP_eis_global'] = grab(p, 'descriptor')
        p = PeptideDescriptor(tmp.name, 'eisenberg')
        p.calculate_moment()
        blocks['mlAMP_eis_moment'] = grab(p, 'descriptor')
        return pd.DataFrame(blocks), True
    except Exception as e:
        print('native modlamp failed ->', repr(e)[:200])
        return pd.DataFrame(index=range(len(seqs))), False


X_ml = pd.DataFrame([modlamp_features(s) for s in seqs])
X_ml_native, MLAMP_OK = modlamp_native(seqs)
if MLAMP_OK and X_ml_native.shape[1] > 0:
    X_ml = pd.concat([X_ml, X_ml_native.reset_index(drop=True)], axis=1)
MODLAMP_COLS = list(X_ml.columns)
X_ml = X_ml.replace([np.inf, -np.inf], np.nan)

print('modlamp matrix:', X_ml.shape, '| native modlamp used:', MLAMP_OK)
print(X_ml.iloc[:5, :12].round(3).to_string())
if 'charge' in [str(c).lower() for c in X_ml.columns]:
    print('net-charge columns present:', [c for c in X_ml.columns if 'charge' in str(c).lower()])


# 8. Feature integration 2.3.3 - the modelling matrices

* **Descriptors** - ProtParam + modlAMP (interpretable, `d << n`)
* **ESM-2** - pooled language-model vectors (1280-d), compressed to `SVD_COMPONENTS` *inside* every CV fold
* **Combined** - both blocks concatenated, i.e. the feature set the proposal feeds to the Random Forest

`SafeSVD` clamps the number of components to `min(k, n_samples-1, n_features-1)` so the pipeline survives the 10/90 split where a training fold holds fewer than 128 samples.


In [ ]:
# @title 8.1 SafeSVD + assembly of the three feature blocks
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD


class SafeSVD(BaseEstimator, TransformerMixin):
    # TruncatedSVD that adapts n_components to tiny folds (small-n regime)

    def __init__(self, n_components=128, random_state=0):
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y=None):
        k = int(max(2, min(self.n_components, X.shape[0] - 1, X.shape[1] - 1)))
        self.svd_ = TruncatedSVD(n_components=k, random_state=self.random_state)
        self.svd_.fit(X)
        self.n_components_ = k
        return self

    def transform(self, X):
        return self.svd_.transform(X)


DESC_COLS = PROTPARAM_COLS + MODLAMP_COLS
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)

X_blocks = {}
if USE_DESCRIPTORS:
    X_blocks['Descriptors'] = X_desc[DESC_COLS].astype(float)
if USE_ESM:
    X_blocks['ESM2'] = X_esm.astype(float)
if USE_DESCRIPTORS and USE_ESM:
    X_blocks['Combined'] = np.hstack([X_desc[DESC_COLS].values.astype(float), X_esm.astype(float)])

y = df['label'].values.astype(int)

print('n =', len(y), '| d_Descriptors =', X_desc.shape[1], '| d_ESM2 =', ESM_DIM)
for k, v in X_blocks.items():
    print('  block %-12s %s' % (k, v.shape))
print('total descriptor features:', len(DESC_COLS))
print('nan cells in descriptor block:', int(np.isnan(X_blocks['Descriptors']).sum()))
print('infinite cells in descriptor block:', int(np.isinf(X_blocks['Descriptors']).sum()))

np.save(os.path.join(WORKDIR, 'y.npy'), y)
for k, v in X_blocks.items():
    np.save(os.path.join(WORKDIR, 'X_%s.npy' % k), v)
df.to_csv(os.path.join(WORKDIR, '02_dataset_final.csv'), index=False)

if SHOW_PCA:
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    D = StandardScaler().fit_transform(X_blocks['Descriptors'])
    P = PCA(n_components=2).fit_transform(D)
    fig, ax = plt.subplots(figsize=(6.2, 5))
    for lab, col, name in [(0, 'tab:orange', 'Inactive'), (1, 'tab:blue', 'Active')]:
        m = y == lab
        ax.scatter(P[m, 0], P[m, 1], s=24, alpha=0.75, c=col, label='%s (n=%d)' % (name, int(m.sum())))
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('Descriptor space (PCA) - %.0f%% of variance' % (100 * PCA(n_components=2).fit(D).explained_variance_ratio_.sum()))
    ax.legend()
    plt.tight_layout()
    plt.show()


# 9. Model zoo (Section 2.4.1)

Every model is a `Pipeline`, so imputation / scaling / SVD are fitted **inside** each CV fold and can never see the test data. Linear and distance-based learners receive the *full* ESM-2 vector after standardisation; tree ensembles receive the SafeSVD-compressed block (at `d = 1280` a tree can only split on noise otherwise). `score_for()` returns an AUROC-compatible score even for models without `predict_proba` (LinearSVC, k-NN, GaussianNB).


In [ ]:
# @title 9.1 Model definitions
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, VotingClassifier)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False
try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    HAVE_LGBM = False
try:
    from catboost import CatBoostClassifier
    HAVE_CAT = True
except Exception:
    HAVE_CAT = False

K_ESM = SVD_COMPONENTS
DESC_DIM = X_blocks['Descriptors'].shape[1]
LINEAR_MODELS = {'LogReg', 'LinearSVC', 'SVC-RBF', 'LDA-shrunk', 'GaussianNB', 'kNN', 'MLP'}
TREE_MODELS = {'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost'}

BASE_MODELS = {}


def _reg():
    return [('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]


def _tree_reg():
    return [('impute', SimpleImputer(strategy='median'))]


def _esm_reg(compress=True):
    steps = [('impute', SimpleImputer(strategy='median'))]
    if compress:
        steps.append(('svd', SafeSVD(n_components=K_ESM, random_state=SEED)))
    steps.append(('scale', StandardScaler()))
    return steps


def _clf_of(fn):
    return fn()['clf'] if isinstance(fn, dict) else fn


BASE_MODELS['LogReg'] = lambda: Pipeline(_reg() + [('clf', LogisticRegression(C=1.0, max_iter=5000, class_weight='balanced'))])
BASE_MODELS['LinearSVC'] = lambda: Pipeline(_reg() + [('clf', LinearSVC(C=0.1, class_weight='balanced', max_iter=20000, random_state=SEED))])
BASE_MODELS['SVC-RBF'] = lambda: Pipeline(_reg() + [('clf', SVC(C=10, gamma='scale', probability=True, class_weight='balanced', random_state=SEED))])
BASE_MODELS['RandomForest'] = lambda: Pipeline(_tree_reg() + [('clf', RandomForestClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED))])
BASE_MODELS['ExtraTrees'] = lambda: Pipeline(_tree_reg() + [('clf', ExtraTreesClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED))])
BASE_MODELS['GradientBoosting'] = lambda: Pipeline(_tree_reg() + [('clf', GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=SEED))])
BASE_MODELS['LDA-shrunk'] = lambda: Pipeline(_reg() + [('clf', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))])
BASE_MODELS['GaussianNB'] = lambda: Pipeline(_reg() + [('clf', GaussianNB())])
BASE_MODELS['kNN'] = lambda: Pipeline(_reg() + [('clf', KNeighborsClassifier(n_neighbors=7, weights='distance'))])
BASE_MODELS['MLP'] = lambda: Pipeline(_reg() + [('clf', MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-2, max_iter=800, early_stopping=True, random_state=SEED))])
if HAVE_XGB:
    BASE_MODELS['XGBoost'] = lambda: Pipeline(_tree_reg() + [('clf', XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, reg_lambda=2.0, min_child_weight=2, eval_metric='logloss', n_jobs=-1, random_state=SEED))])
if HAVE_LGBM:
    BASE_MODELS['LightGBM'] = lambda: Pipeline(_tree_reg() + [('clf', LGBMClassifier(n_estimators=400, num_leaves=15, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, reg_lambda=2.0, min_child_samples=5, class_weight='balanced', n_jobs=-1, random_state=SEED, verbose=-1))])
if HAVE_CAT:
    BASE_MODELS['CatBoost'] = lambda: Pipeline(_tree_reg() + [('clf', CatBoostClassifier(iterations=400, depth=4, learning_rate=0.05, l2_leaf_reg=3, verbose=0, random_seed=SEED, allow_writing_files=False))])


def build_zoo(feature_block, models=None, include_ensemble=True, ensemble_members=('SVC-RBF', 'RandomForest', 'XGBoost', 'LightGBM', 'LogReg')):
    # every returned model is a leakage-free Pipeline for the requested block
    if models is None:
        models = list(BASE_MODELS)
    zoo = {}
    for name in models:
        if name not in BASE_MODELS:
            continue
        clf = BASE_MODELS[name]()['clf']
        if feature_block == 'Descriptors':
            steps = _reg()
        elif feature_block == 'ESM2':
            steps = _esm_reg(compress=(name in TREE_MODELS))
        else:
            if name in TREE_MODELS:
                # descriptors scaled + ESM-2 compressed, then concatenated
                ct = ColumnTransformer([
                    ('desc', Pipeline(_reg()), slice(0, DESC_DIM)),
                    ('esm', Pipeline(_esm_reg(compress=True)), slice(DESC_DIM, None))])
                steps = [('features', ct)]
            else:
                # keep the descriptors: impute + scale descriptors and full ESM-2 together
                steps = _reg()
        zoo[name] = Pipeline(steps + [('clf', clf)])
    if include_ensemble:
        members = [m for m in ensemble_members if m in zoo]
        if len(members) >= 3:
            # every ensemble member must see the same transformation, so borrow the
            # steps of the first non-tree member (plain impute + scale of all features)
            ref = next((m for m in members if m not in TREE_MODELS), members[0])
            steps = [('feat_' + n, clone(s)) for n, s in zoo[ref].steps[:-1]]
            zoo['Voting-soft'] = Pipeline(steps + [('clf', VotingEstimator_safe(members, zoo))])
    return zoo


def VotingEstimator_safe(members, zoo):
    ests = [(m, clone(zoo[m]['clf'])) for m in members]
    return VotingClassifier(estimators=ests, voting='soft', n_jobs=-1)


zoo_preview = build_zoo('Combined')
print('models available:', list(zoo_preview))
print('example Combined RandomForest steps:', [n for n, _ in zoo_preview['RandomForest'].steps])
print('example Combined SVC-RBF steps:', [n for n, _ in zoo_preview['SVC-RBF'].steps])


In [ ]:
# @title 10.1 Metrics
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score, average_precision_score,
                             f1_score, balanced_accuracy_score, confusion_matrix, roc_curve)


def score_for(pipe, X):
    if hasattr(pipe, 'predict_proba'):
        return np.asarray(pipe.predict_proba(X)[:, 1], dtype=float)
    if hasattr(pipe, 'decision_function'):
        return np.asarray(pipe.decision_function(X), dtype=float)
    return np.asarray(pipe.predict(X), dtype=float)


def compute_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    out = {}
    out['acc'] = accuracy_score(y_true, y_pred)
    out['mcc'] = matthews_corrcoef(y_true, y_pred)
    out['f1'] = f1_score(y_true, y_pred, zero_division=0)
    out['bacc'] = balanced_accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out['sens'] = tp / (tp + fn) if (tp + fn) else np.nan
    out['spec'] = tn / (tn + fp) if (tn + fp) else np.nan
    if len(np.unique(y_true)) > 1:
        out['auroc'] = roc_auc_score(y_true, y_score)
        out['auprc'] = average_precision_score(y_true, y_score)
    else:
        out['auroc'] = np.nan
        out['auprc'] = np.nan
    out['y_pred'] = y_pred
    out['y_score'] = y_score
    return out


METRIC_KEYS = ['acc', 'mcc', 'auroc', 'auprc', 'f1', 'bacc', 'sens', 'spec']
print('metrics ready:', METRIC_KEYS)


In [ ]:
# @title 10.2 Split factory (random or cluster-group aware)
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold

groups = df['cluster'].values if GROUP_AWARE_SPLIT else None


def _group_stratified_split(y_, groups_, test_frac, seed, max_rounds=60):
    # greedy cluster assignment that hits the requested test size and keeps both classes present
    n = len(y_)
    target = int(round(test_frac * n))
    want_pos = target * (y_.sum() / n)
    for attempt in range(max_rounds):
        rng = np.random.default_rng(seed * 1000 + attempt)
        uq = np.unique(groups_)
        rng.shuffle(uq)
        chosen, cur, cur_pos = [], 0, 0
        for g in uq:
            if cur >= target:
                break
            sz = int((groups_ == g).sum())
            np_ = int(y_[groups_ == g].sum())
            if cur >= want_pos and np_ == 0 and len(chosen) >= 2:
                continue
            chosen.append(g)
            cur += sz
            cur_pos += np_
        if cur_pos == 0 or cur_pos == y_.sum():
            continue
        mask = np.isin(groups_, chosen)
        te = np.where(mask)[0]
        tr = np.where(~mask)[0]
        if len(te) == 0 or len(tr) == 0:
            continue
        if len(np.unique(y_[te])) == 2 and len(np.unique(y_[tr])) == 2:
            return tr, te
    return train_test_split(np.arange(n), test_size=test_frac, stratify=y_, random_state=seed)


def make_split(train_frac, test_frac, seed):
    # returns (train_idx, test_idx); test_idx is None for the 100/0 case
    idx = np.arange(len(y))
    if test_frac <= 0:
        return idx, None
    if groups is not None:
        tr, te = _group_stratified_split(y, groups, test_frac, seed)
    else:
        tr, te = train_test_split(idx, test_size=test_frac, stratify=y, random_state=seed)
    return np.asarray(tr), np.asarray(te)


print('split type: %s' % ('cluster-group aware' if GROUP_AWARE_SPLIT else 'random'))
print('%9s %8s %7s %11s %10s' % ('split', 'n_train', 'n_test', 'train act%', 'test act%'))
for tf_tr, tf_te in SPLIT_PAIRS:
    tr, te = make_split(tf_tr, tf_te, SEED)
    s_tr = '%.1f%%' % (100 * y[tr].mean())
    s_te = '%.1f%%' % (100 * y[te].mean()) if te is not None else 'CV only'
    print('%4d/%-4d %8d %7s %11s %10s'
          % (int(tf_tr * 100), int(tf_te * 100), len(tr), '-' if te is None else len(te), s_tr, s_te))


In [ ]:
# @title 10.3 Stage A - screening harness (all splits x seeds x models)
from sklearn.base import clone


def evaluate_model_cv(model, X, y_, seed, repeats=CV_REPEATS, folds=CV_FOLDS):
    # used for the 100/0 case: repeated stratified CV, out-of-fold scores pooled
    oof = np.zeros(len(y_), dtype=float)
    cnt = np.zeros(len(y_), dtype=float)
    for rep in range(repeats):
        skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed + 1000 * rep)
        for tr, va in skf.split(X, y_):
            m = clone(model)
            m.fit(X[tr], y_[tr])
            oof[va] += score_for(m, X[va])
            cnt[va] += 1
    return compute_metrics(y_, oof / np.maximum(cnt, 1))


def run_benchmark(feature_block, models, tag='screenA', split_pairs=None, seeds=None):
    X = X_blocks[feature_block]
    zoo_ = build_zoo(feature_block, models=models)
    split_pairs = split_pairs or SPLIT_PAIRS
    seeds = seeds or SCREEN_SEEDS
    rows = []
    t0 = time.time()
    for tf_tr, tf_te in split_pairs:
        for seed in seeds:
            tr, te = make_split(tf_tr, tf_te, seed)
            for mname, proto in zoo_.items():
                try:
                    if te is None:
                        m = evaluate_model_cv(proto, X[tr], y[tr], seed)
                    else:
                        mdl = clone(proto)
                        mdl.fit(X[tr], y[tr])
                        m = compute_metrics(y[te], score_for(mdl, X[te]))
                    rows.append(dict(feature_block=feature_block, model=mname, train_frac=tf_tr,
                                     test_frac=tf_te, seed=seed, n_train=len(tr),
                                     n_test=(0 if te is None else len(te)),
                                     **{k: m[k] for k in METRIC_KEYS}))
                except Exception as e:
                    print('  !! %s on %s/%s seed %s: %s' % (mname, tf_tr, tf_te, seed, repr(e)[:130]))
        print('  split %d/%d finished (%.0fs)' % (int(tf_tr * 100), int(tf_te * 100), time.time() - t0), flush=True)
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(WORKDIR, 'results_%s_%s.csv' % (tag, feature_block)), index=False)
    return res


SCREEN_MODELS = [m for m in ['LogReg', 'SVC-RBF', 'RandomForest', 'ExtraTrees', 'GradientBoosting',
                             'LDA-shrunk', 'GaussianNB', 'kNN', 'MLP', 'XGBoost', 'LightGBM',
                             'CatBoost', 'Voting-soft']
                 if (m in BASE_MODELS or m == 'Voting-soft')]
print('screening models:', SCREEN_MODELS)
print('fits to run:', len(SPLIT_PAIRS) * len(SCREEN_SEEDS) * len(SCREEN_MODELS) * len(CV_REPORT_BLOCKS))


In [ ]:
# @title 10.4 Run the screening benchmark
results = {}
for block in CV_REPORT_BLOCKS:
    if block not in X_blocks:
        continue
    print('=== feature block:', block)
    results[block] = run_benchmark(block, SCREEN_MODELS, tag='screenA')
    print()

res_all = pd.concat(results.values(), ignore_index=True)
res_all.to_csv(os.path.join(WORKDIR, 'results_screenA_all.csv'), index=False)
print('total result rows:', len(res_all))


In [ ]:
# @title 10.5 Results: mean +/- std over seeds, ranked by MCC
def summarise(res, index=('feature_block', 'model')):
    g = res.groupby(list(index))[METRIC_KEYS]
    out = g.agg(['mean', 'std'])
    out.columns = ['%s_%s' % (a, b) for a, b in out.columns]
    return out.sort_values('mcc_mean', ascending=False)


def fmt(summary, cols=('mcc_mean', 'auroc_mean', 'acc_mean')):
    d = summary.copy()
    for c in cols:
        base = c.replace('_mean', '')
        s = d.get(base + '_std', pd.Series(np.nan, index=d.index))
        d[c] = ['%.3f+-%.3f' % (m, 0.0 if pd.isna(sd) else sd) for m, sd in zip(d[c], s)]
    return d[list(cols)]


summary_all = summarise(res_all)
print('=== overall ranking, all splits pooled ===')
print(fmt(summary_all).to_string())
print()
for block in res_all['feature_block'].unique():
    print('=== feature-block ablation: %s ===' % block)
    print(fmt(summarise(res_all[res_all.feature_block == block])).to_string())
    print()


In [ ]:
# @title 10.6 Plots: metric vs split, model x split heatmap, ROC, confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns


def plot_metric_vs_split(res, metric='mcc', top=8, block_tag=''):
    piv = res.pivot_table(index='model', columns='test_frac', values=metric, aggfunc='mean')
    order = res.groupby('model')[metric].mean().sort_values(ascending=False).head(top).index
    order = [m for m in order if m in piv.index]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    for m in order:
        ax[0].plot([c * 100 for c in piv.columns], piv.loc[m], marker='o', label=m)
    ax[0].set_xlabel('test fraction (%)')
    ax[0].set_ylabel(metric.upper())
    ax[0].set_title('%s vs train/test split [%s] (mean of seeds)' % (metric.upper(), block_tag))
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3)
    sns.heatmap(piv.loc[order], annot=True, fmt='.2f', cmap='viridis', ax=ax[1], cbar_kws={'label': metric.upper()})
    ax[1].set_xlabel('test fraction')
    ax[1].set_title('model x split')
    plt.tight_layout()
    plt.show()


main_block = res_all['feature_block'].value_counts().index[0]
sub = res_all[res_all.feature_block == main_block]
plot_metric_vs_split(sub, 'mcc', top=8, block_tag=main_block)
plot_metric_vs_split(sub, 'auroc', top=8, block_tag=main_block)

BEST_MODEL = summary_all.index[0][1]
BEST_BLOCK = summary_all.index[0][0]
print('best pooled MCC:', BEST_BLOCK, '/', BEST_MODEL)

tr, te = make_split(0.80, 0.20, SEED)
Xb = X_blocks[BEST_BLOCK]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for mname in [m for m in SCREEN_MODELS if m != 'Voting-soft']:
    try:
        mdl = clone(build_zoo(BEST_BLOCK, models=[mname], include_ensemble=False)[mname])
        mdl.fit(Xb[tr], y[tr])
        s = score_for(mdl, Xb[te])
        fpr, tpr, _ = roc_curve(y[te], s)
        axes[0].plot(fpr, tpr, lw=1.4, label='%s (AUC %.3f)' % (mname, roc_auc_score(y[te], s)))
    except Exception as e:
        print(mname, repr(e)[:100])
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('false positive rate')
axes[0].set_ylabel('true positive rate')
axes[0].set_title('ROC on the 80/20 hold-out')
axes[0].legend(fontsize=7, loc='lower right')

mdl = clone(build_zoo(BEST_BLOCK, models=[BEST_MODEL] if BEST_MODEL in BASE_MODELS else None,
                      include_ensemble=True)[BEST_MODEL])
mdl.fit(Xb[tr], y[tr])
m = compute_metrics(y[te], score_for(mdl, Xb[te]))
cm = confusion_matrix(y[te], m['y_pred'], labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1],
            xticklabels=['Inactive', 'Active'], yticklabels=['Inactive', 'Active'])
axes[1].set_title('%s: MCC %.3f | AUROC %.3f' % (BEST_MODEL, m['mcc'], m['auroc']))
plt.tight_layout()
plt.show()
print({k: round(m[k], 4) for k in METRIC_KEYS})


# 11. Hyper-parameter optimization (Section 2.4.1)

Randomized search with `CV_FOLDS = 5` cross-validation, executed **only on the training portion of the 80/20 split**, maximising `PRIMARY_METRIC` (`'mcc'`; set it to `'roc_auc'` to optimise AUROC instead). The tuned pipelines are then re-scored on the whole split ladder with 5 seeds. The outer test set is therefore never used for any decision.


In [ ]:
# @title 11.1 Parameter grids + randomized search
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform

PARAM_GRIDS = {
    'RandomForest': {'clf__n_estimators': [300, 500, 800, 1200],
                     'clf__max_depth': [3, 5, 8, 12, None],
                     'clf__min_samples_split': randint(2, 12),
                     'clf__min_samples_leaf': randint(1, 6),
                     'clf__max_features': ['sqrt', 'log2', 0.1, 0.3, 0.6]},
    'ExtraTrees': {'clf__n_estimators': [300, 500, 800, 1200],
                   'clf__max_depth': [3, 5, 8, 12, None],
                   'clf__min_samples_leaf': randint(1, 6),
                   'clf__max_features': ['sqrt', 0.1, 0.3, 0.6]},
    'SVC-RBF': {'clf__C': loguniform(1e-1, 1e3),
                'clf__gamma': ['scale', 'auto', loguniform(1e-4, 1e-1)],
                'clf__class_weight': [None, 'balanced']},
    'LogReg': {'clf__C': loguniform(1e-3, 1e3),
               'clf__class_weight': [None, 'balanced']},
    'XGBoost': {'clf__n_estimators': [200, 400, 800],
                'clf__max_depth': randint(2, 8),
                'clf__learning_rate': loguniform(0.01, 0.3),
                'clf__subsample': uniform(0.6, 0.4),
                'clf__colsample_bytree': uniform(0.2, 0.8),
                'clf__reg_lambda': loguniform(1e-2, 1e2),
                'clf__min_child_weight': randint(1, 6)},
    'LightGBM': {'clf__n_estimators': [200, 400, 800],
                 'clf__num_leaves': randint(7, 63),
                 'clf__learning_rate': loguniform(0.01, 0.3),
                 'clf__subsample': uniform(0.6, 0.4),
                 'clf__colsample_bytree': uniform(0.2, 0.8),
                 'clf__reg_lambda': loguniform(1e-2, 1e2)},
    'CatBoost': {'clf__iterations': [200, 400, 800],
                 'clf__depth': randint(3, 8),
                 'clf__learning_rate': loguniform(0.01, 0.3),
                 'clf__l2_leaf_reg': loguniform(1, 30)},
    'MLP': {'clf__hidden_layer_sizes': [(64,), (128,), (128, 64), (256, 128)],
            'clf__alpha': loguniform(1e-4, 1e0),
            'clf__learning_rate_init': loguniform(1e-4, 1e-2)},
    'GradientBoosting': {'clf__n_estimators': [100, 200, 400],
                         'clf__max_depth': randint(2, 6),
                         'clf__learning_rate': loguniform(0.01, 0.3),
                         'clf__subsample': uniform(0.6, 0.4)},
}


def tune_model(block, model_name, seed=SEED, n_iter=TUNING_N_ITER, train_frac=0.80):
    X = X_blocks[block]
    tr, te = make_split(train_frac, 1.0 - train_frac, seed)
    pipe = build_zoo(block, models=[model_name], include_ensemble=False)[model_name]
    grid = PARAM_GRIDS.get(model_name)
    if grid is None:
        return pipe, None, None
    search = RandomizedSearchCV(pipe, grid, n_iter=n_iter, scoring=PRIMARY_METRIC,
                                cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed),
                                refit=True, n_jobs=-1, random_state=seed)
    if GROUP_AWARE_SPLIT:
        try:
            search.set_params(cv=StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed))
            search.fit(X[tr], y[tr], groups=df['cluster'].values[tr])
        except Exception as e:
            print('grouped CV failed for %s (%s) -> StratifiedKFold' % (model_name, repr(e)[:70]))
            search.set_params(cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed))
            search.fit(X[tr], y[tr])
    else:
        search.fit(X[tr], y[tr])
    return search.best_estimator_, search.best_params_, float(search.best_score_)


TOP_MODELS = [m for m in dict.fromkeys(list(summary_all.index.get_level_values('model'))[:4]) if m in PARAM_GRIDS]
print('tuning these models:', TOP_MODELS, '| block:', BEST_BLOCK, '| metric:', PRIMARY_METRIC)

tuned, tuned_params, tuned_scores = {}, {}, {}
for mname in TOP_MODELS:
    t0 = time.time()
    est, params, score = tune_model(BEST_BLOCK, mname)
    tuned[mname] = est
    tuned_params[mname] = params
    tuned_scores[mname] = score
    print('  %-16s best CV %s = %.4f  (%.0fs)' % (mname, PRIMARY_METRIC, score, time.time() - t0))
    print('     ', params)

pd.DataFrame([{'model': k, 'best_params': str(v), 'cv_%s' % PRIMARY_METRIC: s}
              for k, v, s in zip(tuned_params, tuned_params.values(), tuned_scores.values())]) \
    .to_csv(os.path.join(WORKDIR, 'tuned_params.csv'), index=False)


In [ ]:
# @title 11.2 Re-evaluate the tuned pipelines on the full split ladder (5 seeds)
def run_tuned(pipeline, block, tag):
    X = X_blocks[block]
    rows = []
    t0 = time.time()
    for tf_tr, tf_te in SPLIT_PAIRS:
        for seed in SEEDS:
            tr, te = make_split(tf_tr, tf_te, seed)
            if te is None:
                m = evaluate_model_cv(pipeline, X[tr], y[tr], seed)
            else:
                mdl = clone(pipeline)
                mdl.fit(X[tr], y[tr])
                m = compute_metrics(y[te], score_for(mdl, X[te]))
            rows.append(dict(feature_block=block, model=tag, train_frac=tf_tr, test_frac=tf_te,
                             seed=seed, n_train=len(tr), n_test=(0 if te is None else len(te)),
                             **{k: m[k] for k in METRIC_KEYS}))
        print('  split %d/%d finished (%.0fs)' % (int(tf_tr * 100), int(tf_te * 100), time.time() - t0), flush=True)
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(WORKDIR, 'results_%s_%s.csv' % (tag, block)), index=False)
    return res


res_tuned = {}
for mname, est in tuned.items():
    print('=== tuned %s' % mname)
    res_tuned[mname] = run_tuned(est, BEST_BLOCK, 'tuned-' + mname)
res_tuned_all = pd.concat(res_tuned.values(), ignore_index=True)
res_tuned_all.to_csv(os.path.join(WORKDIR, 'results_tuned_all.csv'), index=False)

print()
print('=== tuned models, MCC per split (mean over 5 seeds) ===')
print(res_tuned_all.pivot_table(index='model', columns='test_frac', values='mcc', aggfunc='mean').round(3).to_string())
print()
print('=== tuned models, AUROC per split (mean over 5 seeds) ===')
print(res_tuned_all.pivot_table(index='model', columns='test_frac', values='auroc', aggfunc='mean').round(3).to_string())


# 12. Final model selection, cross-validated score and bootstrap intervals

The final model is the tuned pipeline with the best mean MCC across the whole split ladder, refitted on all rows. The headline table below is the one to paste into the proposal.


In [ ]:
# @title 12.1 Select the final pipeline and build the headline table
rank_tuned = res_tuned_all.groupby('model')[METRIC_KEYS].mean().sort_values('mcc', ascending=False)
print('=== tuned ranking, mean over all splits and seeds ===')
print(rank_tuned.round(4).to_string())
print()

WINNER = rank_tuned.index[0]
FINAL_PIPELINE = tuned[WINNER.replace('tuned-', '')]
print('selected final model:', WINNER, 'on feature block', BEST_BLOCK)
print('best hyperparameters:', tuned_params[WINNER.replace('tuned-', '')])
print()

headline = res_tuned_all[res_tuned_all.model == WINNER]
tab = headline.groupby(['train_frac', 'test_frac']).agg(
    seeds=('mcc', 'size'),
    ACC=('acc', 'mean'), ACC_sd=('acc', 'std'),
    MCC=('mcc', 'mean'), MCC_sd=('mcc', 'std'),
    AUROC=('auroc', 'mean'), AUROC_sd=('auroc', 'std'),
    AUPRC=('auprc', 'mean'), F1=('f1', 'mean'),
    Sens=('sens', 'mean'), Spec=('spec', 'mean'), BACC=('bacc', 'mean')).round(4)
print('=== headline table: %s ===' % WINNER)
print(tab.to_string())
tab.to_csv(os.path.join(WORKDIR, 'headline_table.csv'))
print()

appendix = res_tuned_all.groupby(['model', 'train_frac', 'test_frac'])[METRIC_KEYS].agg(['mean', 'std']).round(4)
appendix.to_csv(os.path.join(WORKDIR, 'appendix_table.csv'))
print('=== appendix: all tuned models x all splits (MCC / AUROC / ACC) ===')
print(appendix[['mcc', 'auroc', 'acc']].to_string())


In [ ]:
# @title 12.2 Refit on all data, 5-fold CV, bootstrap CIs, save artefacts
import joblib

X_best = X_blocks[BEST_BLOCK]

cv_out = {k: [] for k in ['mcc', 'auroc', 'acc', 'auprc']}
oof = np.zeros(len(y))
oof_cnt = np.zeros(len(y))
for tr, va in StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED).split(X_best, y):
    mdl = clone(FINAL_PIPELINE)
    mdl.fit(X_best[tr], y[tr])
    s = score_for(mdl, X_best[va])
    oof[va] += s
    oof_cnt[va] += 1
    mm = compute_metrics(y[va], s)
    for k in cv_out:
        cv_out[k].append(mm[k])
oof_pred = oof / np.maximum(oof_cnt, 1)
m_oof = compute_metrics(y, oof_pred)

FINAL_PIPELINE.fit(X_best, y)
joblib.dump({'pipeline': FINAL_PIPELINE, 'feature_block': BEST_BLOCK, 'esm_model': ESM_MODEL,
             'esm_pool': ESM_POOL, 'desc_cols': DESC_COLS, 'cluster_identity': CLUSTER_IDENTITY,
             'std_scaler': 'in-pipeline'},
            os.path.join(WORKDIR, 'final_model.joblib'))

print('=== %d-fold CV of the final pipeline (n = %d) ===' % (CV_FOLDS, len(y)))
for k, v in cv_out.items():
    print('  %-6s %.3f +- %.3f' % (k, float(np.nanmean(v)), float(np.nanstd(v))))
print()
print('=== pooled out-of-fold metrics (single 5-fold run) ===')
print({k: round(m_oof[k], 4) for k in METRIC_KEYS})
print()
print(pd.DataFrame({'y_true': y, 'p_active': oof_pred}).assign(correct=lambda d: (d.p_active >= 0.5).astype(int) == d.y_true).head(10).to_string(index=False))

rng = np.random.default_rng(SEED)
boot = {k: [] for k in ['mcc', 'auroc', 'acc']}
s_full = score_for(FINAL_PIPELINE, X_best)
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y), len(y))
    if len(np.unique(y[idx])) < 2:
        continue
    mb = compute_metrics(y[idx], s_full[idx])
    for k in boot:
        boot[k].append(mb[k])
print()
print('=== in-sample bootstrap 95% CIs (optimistic - sanity check only) ===')
for k, v in boot.items():
    print('  %-6s %.3f - %.3f' % (k, float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))))


# 13. SHAP-guided interpretability (Section 2.4.2)

Three views of the same Random Forest:

1. **TreeSHAP on the descriptor block** - a human readable ranking (net charge, hydrophobic moment, GRAVY, aliphatic index, instability index ...) with a beeswarm plot showing the *direction* of every effect. This is the panel that supports the chemistry narrative in the proposal.
2. **TreeSHAP on the ESM-2 block** - ranks the 1280 abstract embedding dimensions, i.e. which language-model features the model relies on.
3. **Per-residue attribution mapped onto sequence positions** - because the pooled ESM-2 vector is the mean of per-residue vectors, the contribution of residue *i* is isolated by replacing its embedding with the dataset-average residue embedding and measuring the drop in predicted activity. This is the mapping from embedding dimension back to physical sequence position requested in the proposal.


In [ ]:
# @title 13.1 TreeSHAP global explanations
import shap
from sklearn.inspection import permutation_importance

TREE_TYPES = ('RandomForestClassifier', 'ExtraTreesClassifier', 'GradientBoostingClassifier',
              'LGBMClassifier', 'XGBClassifier', 'CatBoostClassifier')


def shap_explain(pipeline, X, feature_names=None, sample=300, seed=SEED, title=''):
    Xt = np.asarray(pipeline[:-1].transform(X))
    clf = pipeline['clf']
    if feature_names is None or len(feature_names) != Xt.shape[1]:
        # the ESM-2 block is compressed by SafeSVD, so name the *transformed* columns
        feature_names = ['%s_f%d' % (title.split()[0].lower(), i) for i in range(Xt.shape[1])]
        print('%s: design matrix is %d columns wide -> using %s'
              % (title, Xt.shape[1], 'SVD component' if Xt.shape[1] < X.shape[1] else 'raw feature'))
    if type(clf).__name__ in TREE_TYPES:
        expl = shap.TreeExplainer(clf)
        sv = expl.shap_values(Xt[:sample])
        base = float(np.ravel(expl.expected_value)[-1]) if np.size(expl.expected_value) > 1 else float(expl.expected_value)
    else:
        # non-tree fallback: permutation importance on the same design matrix
        pi = permutation_importance(clf, Xt, y, n_repeats=10, random_state=seed, scoring='roc_auc')
        imp = pd.Series(pi.importances_mean, index=feature_names).sort_values(ascending=False)
        print('--- %s: %s is not tree based -> permutation importance shown instead ---' % (title, type(clf).__name__))
        print(imp.head(25).to_string())
        plt.figure(figsize=(8, 6))
        plt.barh(imp.head(25).index[::-1], imp.head(25).values[::-1], color='tab:blue')
        plt.xlabel('permutation importance (AUROC drop)')
        plt.title('Feature importance: %s' % title)
        plt.tight_layout()
        plt.show()
        return None, imp
    if isinstance(sv, list):
        sv = sv[-1]
    print('--- SHAP for %s | base value (expected output) = %.4f ---' % (title, base))
    shap.summary_plot(sv, Xt[:sample], feature_names=feature_names, max_display=25, show=False)
    plt.title('TreeSHAP attribution: %s' % title)
    plt.tight_layout()
    plt.show()
    imp = pd.Series(np.abs(sv).mean(axis=0), index=feature_names).sort_values(ascending=False)
    print('mean(|SHAP|), top 15:')
    print(imp.head(15).to_string())
    return sv, imp


shap_tables = {}

rf_proto = tuned.get('RandomForest', build_zoo(BEST_BLOCK, models=['RandomForest'],
                                               include_ensemble=False)['RandomForest'])


def retarget(pipe, block):
    # keep the tuned classifier, swap in the preprocessing of another feature block
    base = build_zoo(block, models=['RandomForest'], include_ensemble=False)['RandomForest']
    return Pipeline(list(base.steps[:-1]) + [('clf', clone(pipe['clf']))])


# (a) descriptor block
try:
    pipe_d = retarget(rf_proto, 'Descriptors')
    pipe_d.fit(X_blocks['Descriptors'], y)
    sv_d, imp_d = shap_explain(pipe_d, X_blocks['Descriptors'], DESC_COLS, title='Descriptors')
    shap_tables['descriptors'] = imp_d
except Exception as e:
    print('descriptor SHAP failed:', repr(e)[:200])

# (b) ESM-2 block (compressed by SafeSVD -> the SHAP values belong to SVD components)
if USE_ESM:
    try:
        pipe_e = retarget(rf_proto, 'ESM2')
        pipe_e.fit(X_blocks['ESM2'], y)
        sv_e, imp_e = shap_explain(pipe_e, X_blocks['ESM2'], title='ESM2')
        shap_tables['esm2'] = imp_e
        # map the most important SVD components back to the original 1280 ESM-2 dimensions
        comps = pipe_e['svd'].svd_.components_
        print()
        print('=== top SVD components mapped back to the original ESM-2 dimensions ===')
        for rank, fname in enumerate(imp_e.head(8).index):
            k = int(fname.split('_f')[-1])
            load = comps[k]
            top_dims = np.argsort(-np.abs(load))[:8]
            print('%-16s (explained variance %.3f) <- dims %s'
                  % (fname, pipe_e['svd'].svd_.explained_variance_ratio_[k], list(top_dims)))
    except Exception as e:
        print('ESM-2 SHAP failed:', repr(e)[:200])

if shap_tables:
    pd.concat(shap_tables, axis=1).to_csv(os.path.join(WORKDIR, 'shap_importance.csv'))
    print('\nsaved shap_importance.csv')


In [ ]:
# @title 13.2 Per-residue attribution: embedding dimensions -> sequence positions
import torch
from transformers import AutoTokenizer, AutoModel


@torch.no_grad()
def esm2_per_residue(seq, model_name=ESM_MODEL, max_len=ESM_MAX_LEN, device=DEVICE):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).eval().to(device)
    if device == 'cuda':
        model.half()
    enc = tok([seq], return_tensors='pt', truncation=True, max_length=max_len)
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc).last_hidden_state[0].float().cpu().numpy()
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()
    return out[1:1 + min(len(seq), max_len - 2)]      # strip <cls>/<eos>


class PerResidueExplainer:
    # occlusion of residue i: swap its ESM-2 vector for the dataset-average residue vector
    def __init__(self, pipeline, block, base_X, n_avg=40):
        self.pipe = pipeline
        self.block = block
        self.d_dim = X_blocks['Descriptors'].shape[1] if block == 'Combined' else 0
        vecs = np.vstack([esm2_per_residue(s) for s in df['sequence'].tolist()[:n_avg]])
        self.avg_residue = vecs.mean(axis=0)
        self.base = np.asarray(base_X, dtype=float)

    def explain(self, row_idx):
        seq = df['sequence'].iloc[row_idx]
        per_res = esm2_per_residue(seq)
        p0 = float(score_for(self.pipe, self.base[[row_idx]])[0])
        contrib = np.zeros(len(per_res))
        for i in range(len(per_res)):
            Xmod = self.base[[row_idx]].copy()
            if self.block == 'Combined':
                Xmod[0, self.d_dim:] = Xmod[0, self.d_dim:] - per_res[i] + self.avg_residue[i]
            else:
                Xmod[0, :] = Xmod[0, :] - per_res[i] + self.avg_residue[i]
            contrib[i] = p0 - float(score_for(self.pipe, Xmod)[0])
        return seq, p0, contrib


if ESM_POOL != 'mean':
    print('per-residue attribution requires ESM_POOL == "mean" (mean-pooled vector = mean of residue vectors)')
else:
    FINAL_PIPELINE.fit(X_blocks[BEST_BLOCK], y)
    explainer = PerResidueExplainer(FINAL_PIPELINE, BEST_BLOCK, X_blocks[BEST_BLOCK])
    demo_idx = list(np.argsort(-y)[:3]) + list(np.argsort(y)[:3])   # 3 actives + 3 inactives
    records = []
    for i in demo_idx:
        seq, p0, c = explainer.explain(i)
        print()
        print('peptide %s | label=%d | len=%d | p(active)=%.3f' % (df['pep_id'].iloc[i], y[i], len(seq), p0))
        print('seq     :', seq)
        print('contrib :', np.array2string(c, precision=4, max_line_width=250, separator=', '))
        records.append((df['pep_id'].iloc[i], int(y[i]), seq, p0, c))

    fig, axes = plt.subplots(len(records), 1, figsize=(14, 2.3 * len(records)), sharex=False)
    axes = np.atleast_1d(axes)
    for ax, (pid, lab, seq, p0, c) in zip(axes, records):
        ax.bar(range(len(c)), c, color=['tab:red' if v < 0 else 'tab:blue' for v in c])
        ax.set_title('%s | label=%d | p(active)=%.2f | blue raises activity, red suppresses'
                     % (pid, lab, p0), fontsize=9)
        ax.set_ylabel('delta p')
        ax.set_xticks(range(len(seq)))
        ax.set_xticklabels(list(seq), fontsize=7)
        ax.axhline(0, color='k', lw=0.8)
    plt.tight_layout()
    plt.show()


In [ ]:
# @title 13.3 Where along the sequence does the model look?
rows = []
for pid, lab, seq, p0, c in records:
    for i, (aa, v) in enumerate(zip(seq, c)):
        rows.append(dict(pep=pid, label=lab, pos=i, rel_pos=i / max(1, len(seq) - 1), aa=aa, contrib=v))
pos_df = pd.DataFrame(rows)
pos_df['bin'] = pd.cut(pos_df['rel_pos'], bins=10, labels=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
sns.barplot(data=pos_df, x='bin', y='contrib', ax=axes[0], color='tab:blue', errorbar=None)
axes[0].set_xlabel('relative position decile (0 = N-terminus, 9 = C-terminus)')
axes[0].set_title('mean signed contribution by position')
axes[0].axhline(0, color='k', lw=0.8)

aa_tab = pos_df.groupby('aa')['contrib'].agg(['mean', 'count'])
aa_tab['mean_abs'] = pos_df.groupby('aa')['contrib'].apply(lambda s: s.abs().mean())
aa_tab = aa_tab[aa_tab['count'] >= 2].sort_values('mean_abs', ascending=False).head(15)
sns.barplot(data=aa_tab, x='aa', y='mean_abs', ax=axes[1], color='tab:orange')
axes[1].set_xlabel('residue type')
axes[1].set_title('mean |contribution| by residue type')
axes[1].tick_params(axis='x', rotation=45)

sns.boxplot(data=pos_df, x='label', y='contrib', ax=axes[2], palette=['tab:orange', 'tab:blue'])
axes[2].set_xticklabels(['Inactive (0)', 'Active (1)'])
axes[2].set_title('per-residue contribution by activity class')
axes[2].axhline(0, color='k', lw=0.8)
plt.tight_layout()
plt.show()

pos_df.to_csv(os.path.join(WORKDIR, 'per_residue_attribution.csv'), index=False)
print(pos_df.groupby('bin')['contrib'].mean().round(4).to_string())


# 14. Save artefacts and score new peptides

`final_model.joblib` holds the fitted pipeline together with the metadata needed to reproduce its features (ESM-2 checkpoint, pooling, descriptor column order). The helper below takes raw peptide strings, runs the exact same feature extraction and returns a probability of activity.


In [ ]:
# @title 14.1 Inference helper
bundle = joblib.load(os.path.join(WORKDIR, 'final_model.joblib'))
PIPE = bundle['pipeline']
BLOCK = bundle['feature_block']
DESC_COLS_SAVED = bundle['desc_cols']


def build_features(sequences):
    # identical feature pipeline as training, but for unseen sequences
    prot = pd.DataFrame([protparam_features(s) for s in sequences])[PROTPARAM_COLS]
    ml = pd.DataFrame([modlamp_features(s) for s in sequences])
    if MLAMP_OK:
        nat, ok = modlamp_native(list(sequences))
        if ok and nat.shape[1] == X_ml_native.shape[1]:
            ml = pd.concat([ml, nat.reset_index(drop=True)], axis=1)
        else:
            print('warning: native modlamp columns unavailable for the new sequences')
    ml = ml[MODLAMP_COLS]
    if bundle.get('esm_model'):
        XE = esm2_embed(list(sequences), model_name=bundle['esm_model'],
                        pool=bundle['esm_pool'], verbose=False)
    else:
        XE = np.zeros((len(sequences), ESM_DIM))
    desc = np.hstack([prot[PROTPARAM_COLS].values.astype(float), ml[MODLAMP_COLS].values.astype(float)])
    if BLOCK == 'Descriptors':
        return desc
    if BLOCK == 'ESM2':
        return XE
    return np.hstack([desc, XE])


def predict_peptides(sequences, threshold=0.5):
    seqs_ = [clean_seq(s)[0] for s in sequences]
    Xn = build_features(seqs_)
    p = score_for(PIPE, Xn)
    return pd.DataFrame({
        'sequence': seqs_,
        'length': [len(s) for s in seqs_],
        'p_active': p,
        'prediction': np.where(p >= threshold, 'Active', 'Inactive'),
        'instability_index': [protparam_features(s)['instability_index'] for s in seqs_],
        'net_charge': [modlamp_features(s)['ml_net_charge_pH7'] for s in seqs_],
        'hydrophobic_moment': [modlamp_features(s)['ml_hydrophobic_moment'] for s in seqs_],
        'gravy': [protparam_features(s)['gravy'] for s in seqs_],
    })


demo_new = ['KKLIKVWAKGFKKAKKLFKGIG', 'GSGSGSGSLKKIFKKPMVIGVTIPF', 'FLSLIPHAINAVSAIAKHN']
try:
    out = predict_peptides(demo_new)
    print(out.round(4).to_string(index=False))
    truth = df.set_index('sequence')['label']
    for s in demo_new:
        s = clean_seq(s)[0]
        if s in truth.index:
            print('observed label for %s: %s' % (s, 'Active' if truth[s] == 1 else 'Inactive'))
except Exception as e:
    print('inference demo failed:', repr(e)[:300])

print()
print('artefacts in', WORKDIR)
for f in sorted(os.listdir(WORKDIR)):
    print('  ', f, round(os.path.getsize(os.path.join(WORKDIR, f)) / 1024, 1), 'KB')


# 15. (Optional) fine-tune ESM-2 with a classification head

Frozen ESM-2 embeddings are the cheap option analysed above. If the GPU allows it, replacing the last `k` transformer blocks (or injecting LoRA adapters) and training a linear head is the option with the highest ceiling. Keep it clearly labelled as an extension, use the same cluster-grouped splits, and report at least 3 seeds, because a 600-row dataset makes single-run fine-tuned numbers unreliable.

Set `RUN_FINETUNE = True` after the main pipeline results have been saved.


In [ ]:
# @title 15.1 LoRA / partial fine-tuning of ESM-2
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel

RUN_FINETUNE = False
FINETUNE_MODEL = 'facebook/esm2_t30_150M_UR50D'    # t33_650M if VRAM allows
FT_EPOCHS = 6
FT_LR = 1e-5
FT_BATCH = 8
FREEZE_ALL_BUT_LAST = 4                              # unfreeze the last k encoder layers

if RUN_FINETUNE:
    from contextlib import nullcontext
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    AMP = True
    try:
        AMP_AUTOCAST = torch.amp.autocast('cuda')
        AMP_SCALER = torch.amp.GradScaler('cuda') if dev == 'cuda' else None
    except Exception:
        AMP = False
        AMP_AUTOCAST = nullcontext
        AMP_SCALER = None
    tok = AutoTokenizer.from_pretrained(FINETUNE_MODEL)
    backbone = AutoModel.from_pretrained(FINETUNE_MODEL).to(dev)
    n_layers = len(backbone.encoder.layer)
    for p in backbone.parameters():
        p.requires_grad = False
    for layer in backbone.encoder.layer[n_layers - FREEZE_ALL_BUT_LAST:]:
        for p in layer.parameters():
            p.requires_grad = True
    head = nn.Linear(backbone.config.hidden_size, 2).to(dev)

    def make_batch(sequences, labels=None):
        enc = tok(list(sequences), return_tensors='pt', padding=True, truncation=True, max_length=ESM_MAX_LEN)
        enc = {k: v.to(dev) for k, v in enc.items()}
        lab = torch.tensor(labels, dtype=torch.long, device=dev) if labels is not None else None
        return enc, lab

    def forward_logits(enc):
        out = backbone(**enc).last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return head(pooled)

    tr, te = make_split(0.80, 0.20, SEED)
    opt = torch.optim.AdamW([p for p in backbone.parameters() if p.requires_grad] + list(head.parameters()),
                            lr=FT_LR, weight_decay=0.01)
    lossf = nn.CrossEntropyLoss()
    scaler = AMP_SCALER

    best_score, best_state = -1, None
    for ep in range(FT_EPOCHS):
        backbone.train(); head.train()
        order = np.random.permutation(tr)
        for s in range(0, len(order), FT_BATCH):
            b = order[s:s + FT_BATCH]
            enc, lab = make_batch([seqs[i] for i in b], y[b])
            with AMP_AUTOCAST():
                loss = lossf(forward_logits(enc), lab)
            opt.zero_grad()
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()
        backbone.eval(); head.eval()
        preds = []
        with torch.no_grad():
            for s in range(0, len(te), FT_BATCH):
                b = te[s:s + FT_BATCH]
                enc, _ = make_batch([seqs[i] for i in b])
                with AMP_AUTOCAST():
                    p = torch.softmax(forward_logits(enc), dim=-1)[:, 1]
                preds.append(p.float().cpu().numpy())
        mm = compute_metrics(y[te], np.concatenate(preds))
        print('epoch %d | loss %.4f | MCC %.3f | AUROC %.3f' % (ep + 1, float(loss), mm['mcc'], mm['auroc']))
        if mm['mcc'] > best_score:
            best_score = mm['mcc']
            best_state = ({k: v.detach().cpu().clone() for k, v in backbone.state_dict().items()},
                          {k: v.detach().cpu().clone() for k, v in head.state_dict().items()})
    torch.save({'backbone': best_state[0], 'head': best_state[1], 'model_name': FINETUNE_MODEL},
               os.path.join(WORKDIR, 'finetuned_esm2.pt'))
    print('best fine-tuned MCC on 80/20:', round(best_score, 4))
else:
    print('RUN_FINETUNE = False -> cell skipped (results above are from the frozen-embedding pipeline)')


# 16. Reporting checklist for the proposal

* Report **MCC** as the primary metric, with AUROC + 95% CI, accuracy, balanced accuracy, sensitivity and specificity.
* Always state the **split type**: random vs CD-HIT 90% cluster-grouped, and whether exact duplicates were collapsed.
* Show the **ablation** (Descriptors / ESM-2 / Combined). If Descriptors is within noise of Combined, that is a finding, not a failure.
* Report **mean +- std over seeds**, never a single run.
* Justify the final model family in the discussion using the small-n argument (capacity control, dimensionality reduction, ensemble averaging).
* Use SHAP section 13.1 to explain *why* a peptide is predicted active, and 13.2-13.3 to point at the residues that drive the prediction.
